[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongzoolee/hidden-bites/blob/main/notebooks/hb-score-calculation.ipynb)

# HB Score Calculation

이 노트북은 Google Maps 리뷰를 Hidden Bites의 factor 관점으로 다시 점수화합니다. 기본 아이디어는 리뷰 본문과 factor 설명문 사이의 NLI entailment probability를 의미적 proximity로 보고, 그 proximity만큼 원래 별점이 해당 factor score에 기여하도록 만드는 것입니다.

최종 web-app에서는 x-axis dropdown factor를 선택하고, y-axis에는 식당별 `hb_score`를 `0.00~5.00` 범위로 그릴 수 있습니다.

## 0. 실행 환경 준비

Colab에서는 아래 셀이 필요한 패키지를 설치합니다. 로컬에서 이미 설치되어 있다면 빠르게 지나갑니다. NLI 모델은 제공된 Colab 파일의 방식과 동일하게 `sentence_transformers.cross_encoder.CrossEncoder`를 사용합니다.

In [ ]:
%pip install -q pandas numpy scipy sentence-transformers transformers torch tqdm

## 1. 경로와 실행 모드

기본값은 `smoke`입니다. 로컬 검증에서는 2개 식당, 식당별 20개 리뷰만 scoring합니다. Colab GPU에서 전체 산출물을 만들 때는 환경변수 `HB_SCORE_RUN_MODE=full` 또는 아래 `RUN_MODE = "full"`로 바꿔 실행합니다.

- `smoke`: `/tmp/hidden-bites-hb-score-smoke`에 검증용 결과 저장
- `full`: `datasets/derived/`에 web-app용 결과 저장

In [ ]:
from pathlib import Path
import os
import subprocess

RUN_MODE = os.environ.get("HB_SCORE_RUN_MODE", "smoke")
SMOKE_RESTAURANT_LIMIT = int(os.environ.get("HB_SCORE_SMOKE_RESTAURANT_LIMIT", "2"))
SMOKE_REVIEWS_PER_RESTAURANT = int(os.environ.get("HB_SCORE_SMOKE_REVIEWS_PER_RESTAURANT", "20"))
REPO_URL = "https://github.com/dongzoolee/hidden-bites.git"


def is_colab_runtime():
    try:
        import google.colab
        return True
    except ModuleNotFoundError:
        return False


if is_colab_runtime() and not Path("hidden-bites").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

if Path("datasets/google-maps-reviews-2026-05-16").exists():
    PROJECT_ROOT = Path(".").resolve()
elif Path("hidden-bites/datasets/google-maps-reviews-2026-05-16").exists():
    PROJECT_ROOT = Path("hidden-bites").resolve()
else:
    PROJECT_ROOT = Path("/Users/dongzoolee/Projects/hidden-bites").resolve()

DATASET_DIR = PROJECT_ROOT / "datasets" / "google-maps-reviews-2026-05-16"
OUTPUT_DIR = PROJECT_ROOT / "datasets" / "derived"
SMOKE_OUTPUT_DIR = Path("/tmp/hidden-bites-hb-score-smoke")
ACTIVE_OUTPUT_DIR = OUTPUT_DIR if RUN_MODE == "full" else SMOKE_OUTPUT_DIR

print(f"RUN_MODE={RUN_MODE}")
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"DATASET_DIR={DATASET_DIR}")
print(f"ACTIVE_OUTPUT_DIR={ACTIVE_OUTPUT_DIR}")

## 2. 기본 import

In [ ]:
from collections import defaultdict
from datetime import datetime, timezone
import gzip
import json
import math
import re

import numpy as np
import pandas as pd
from scipy.special import softmax
from sentence_transformers.cross_encoder import CrossEncoder
from tqdm.auto import tqdm

## 3. Factor schema

Factor는 web-app dropdown 후보로 바로 쓸 수 있도록 10개로 고정합니다. 각 factor는 3개의 English hypothesis를 가집니다. NLI는 리뷰 premise가 각 hypothesis를 얼마나 entail하는지 계산합니다.

In [ ]:
FACTORS = [
    {
        "id": "taste",
        "label": "Taste",
        "structured_rating_label": "음식",
        "hypotheses": [
            "The reviewer comments on the taste or flavor of the food.",
            "The reviewer says the food, dishes, or menu items were good or bad.",
            "The reviewer mentions specific dishes, ingredients, or cooking quality.",
        ],
    },
    {
        "id": "service",
        "label": "Service",
        "structured_rating_label": "서비스",
        "hypotheses": [
            "The reviewer comments on the staff or customer service.",
            "The reviewer says employees were friendly, helpful, slow, or rude.",
            "The reviewer mentions orders, refills, guidance, or service requests.",
        ],
    },
    {
        "id": "value",
        "label": "Value",
        "structured_rating_label": None,
        "hypotheses": [
            "The reviewer comments on the price or value for money.",
            "The reviewer says the meal was worth the price or not worth the price.",
            "The reviewer mentions expensive prices, cheap prices, or affordability.",
        ],
    },
    {
        "id": "atmosphere",
        "label": "Atmosphere",
        "structured_rating_label": "분위기",
        "hypotheses": [
            "The reviewer comments on the atmosphere, interior, or dining space.",
            "The reviewer says the restaurant was comfortable, stylish, noisy, quiet, crowded, or spacious.",
            "The reviewer mentions seating, tables, ambience, or the physical space.",
        ],
    },
    {
        "id": "accessibility",
        "label": "Accessibility",
        "structured_rating_label": None,
        "hypotheses": [
            "The reviewer comments on the restaurant's location, transportation, or parking.",
            "The reviewer mentions a station, neighborhood, mall, landmark, or nearby place.",
            "The reviewer says the restaurant was easy or difficult to find or reach.",
        ],
    },
    {
        "id": "wait_queue",
        "label": "Wait/Queue",
        "structured_rating_label": None,
        "hypotheses": [
            "The reviewer waited in line before being seated at the restaurant.",
            "The reviewer mentions a reservation or waiting time at the restaurant.",
            "The reviewer says the restaurant had a queue or a delay before entering.",
        ],
    },
    {
        "id": "visit_occasion",
        "label": "Visit Occasion",
        "structured_rating_label": None,
        "hypotheses": [
            "The reviewer mentions visiting with family, friends, a partner, coworkers, or alone.",
            "The reviewer mentions a date, birthday, trip, gathering, company meal, or special occasion.",
            "The reviewer explains why or with whom they visited the restaurant.",
        ],
    },
    {
        "id": "portion",
        "label": "Portion",
        "structured_rating_label": None,
        "hypotheses": [
            "The reviewer comments on portion size or serving amount.",
            "The reviewer says the portions were generous, small, filling, or insufficient.",
            "The reviewer mentions sharing food, being full, or getting a lot of food.",
        ],
    },
    {
        "id": "cleanliness",
        "label": "Cleanliness",
        "structured_rating_label": None,
        "hypotheses": [
            "The reviewer comments on cleanliness, hygiene, or tidiness.",
            "The reviewer says the restaurant was clean, neat, sanitary, dirty, or poorly maintained.",
            "The reviewer mentions clean space, clean tableware, hygiene, or messiness.",
        ],
    },
    {
        "id": "signature_uniqueness",
        "label": "Signature/Uniqueness",
        "structured_rating_label": None,
        "hypotheses": [
            "The reviewer mentions a signature dish, representative menu item, or famous dish.",
            "The reviewer says the restaurant had a unique menu, special concept, or distinctive experience.",
            "The reviewer mentions a viral, memorable, unusual, or must-try item.",
        ],
    },
]

FACTOR_IDS = [factor["id"] for factor in FACTORS]
FACTOR_LABELS = {factor["id"]: factor["label"] for factor in FACTORS}
STRUCTURED_RATING_BY_FACTOR = {factor["id"]: factor["structured_rating_label"] for factor in FACTORS}


## 4. Google Maps 리뷰 로드

장소별 final JSON 50개를 읽고, 리뷰 한 개가 한 행이 되도록 펼칩니다. `.partial.json`과 `run-metadata.json`은 제외합니다.

In [ ]:
def final_review_files(dataset_dir: Path):
    return sorted(
        path
        for path in dataset_dir.glob("*.json")
        if not path.name.endswith(".partial.json") and path.name != "run-metadata.json"
    )


def as_text(value):
    if isinstance(value, str):
        return value.strip()
    return ""


def normalize_space(value):
    return re.sub(r"\s+", " ", as_text(value)).strip()


def parse_structured_rating(meal_type_text, label):
    if not isinstance(meal_type_text, list):
        return np.nan
    pattern = re.compile(rf"^{re.escape(label)}\s*:\s*([1-5])$")
    for item in meal_type_text:
        match = pattern.match(str(item).strip())
        if match:
            return float(match.group(1))
    return np.nan


def review_context_text(review):
    text = normalize_space(review.get("text"))
    if not text:
        text = normalize_space(review.get("raw_text"))
    meal_type_text = review.get("meal_type_text") if isinstance(review.get("meal_type_text"), list) else []
    meal_text = " ".join(str(item).strip() for item in meal_type_text if str(item).strip())
    return normalize_space(" ".join(part for part in [text, meal_text] if part))


def build_reviews_df(dataset_dir: Path):
    rows = []
    for file_path in final_review_files(dataset_dir):
        with file_path.open("r", encoding="utf-8") as file:
            payload = json.load(file)
        metadata = payload.get("metadata", {})
        place = metadata.get("place", {})
        reviews = payload.get("reviews", []) if isinstance(payload.get("reviews"), list) else []
        collected_review_count = len(reviews)
        for review_index, review in enumerate(reviews, start=1):
            meal_type_text = review.get("meal_type_text") if isinstance(review.get("meal_type_text"), list) else []
            rating = review.get("rating")
            rows.append(
                {
                    "review_uid": f"{metadata.get('place_rank', 0):03d}-{review_index:05d}",
                    "source_review_id": review.get("source_review_id"),
                    "place_rank": metadata.get("place_rank"),
                    "place_id": place.get("place_id") or review.get("place_id"),
                    "place_name": place.get("name") or review.get("place_name"),
                    "formatted_address": place.get("formatted_address"),
                    "google_maps_uri": place.get("google_maps_uri") or review.get("google_maps_uri"),
                    "place_rating": place.get("rating"),
                    "user_rating_count": place.get("user_rating_count"),
                    "collected_review_count": collected_review_count,
                    "collection_status": metadata.get("status"),
                    "review_index": review_index,
                    "rating": float(rating) if isinstance(rating, (int, float)) else np.nan,
                    "relative_time": review.get("relative_time"),
                    "text": as_text(review.get("text")),
                    "meal_type_text": meal_type_text,
                    "context_text": review_context_text(review),
                    "food_rating": parse_structured_rating(meal_type_text, "음식"),
                    "service_rating": parse_structured_rating(meal_type_text, "서비스"),
                    "atmosphere_rating": parse_structured_rating(meal_type_text, "분위기"),
                }
            )
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No final Google Maps review JSON files found in {dataset_dir}")
    return df


reviews_df = build_reviews_df(DATASET_DIR)
summary = {
    "restaurants": int(reviews_df["place_id"].nunique()),
    "reviews": int(len(reviews_df)),
    "non_empty_context_reviews": int(reviews_df["context_text"].astype(bool).sum()),
    "final_json_files": int(len(final_review_files(DATASET_DIR))),
}
summary

## 5. Factor 후보 coverage 확인

NLI를 돌리기 전에 간단한 keyword scan으로 factor 후보가 전체 데이터에서 어느 정도 등장하는지 확인합니다. 이 값은 최종 score가 아니라 factor 후보의 데이터 커버리지 점검용입니다.

In [ ]:
KEYWORD_PATTERNS = {
    "taste": r"맛있|맛집|음식|요리|메뉴|고기|소스|신선|식감|풍미|짜장|라멘|갈비|순대|파스타|국수|곱창|닭갈비|두부",
    "service": r"친절|서비스|응대|직원|사장|서빙|안내|설명|배려|리필|주문",
    "value": r"가성비|가격|비싸|저렴|양이|양도|푸짐|만원|₩|혜자|합리|무한",
    "atmosphere": r"분위기|인테리어|깔끔|쾌적|조용|시끄|소음|넓|자리|매장|공간|테이블|데이트|고급|세련",
    "accessibility": r"위치|역|가까|근처|찾기|주차|교통|접근|코엑스|홍대|명동|성수|강남|이태원|공덕|망원|잠실|압구정|마포|종로|을지로",
    "wait_queue": r"웨이팅|대기|줄|예약|바로\s*입장|기다|오래|금방|회전|캐치테이블",
    "visit_occasion": r"데이트|가족|친구|회식|혼밥|혼자|커플|부모님|아이|여행|관광|외국|기념|생일|모임|직장|동료",
    "portion": r"양이|양도|양은|푸짐|많아|많고|배부|넉넉|든든",
    "cleanliness": r"깨끗|청결|위생|깔끔|정갈|쾌적",
    "signature_uniqueness": r"시그니처|특별|특색|유명|대표|원픽|차별|인생|처음|색다|독특|핫플|흑백요리사",
}

coverage_rows = []
coverage_text = reviews_df["context_text"].fillna("")
for factor in FACTORS:
    mask = coverage_text.str.contains(KEYWORD_PATTERNS[factor["id"]], regex=True, case=False, na=False)
    coverage_rows.append(
        {
            "factor_id": factor["id"],
            "factor_label": factor["label"],
            "keyword_review_count": int(mask.sum()),
            "keyword_review_share": round(float(mask.mean()), 4),
            "place_count": int(reviews_df.loc[mask, "place_id"].nunique()),
        }
    )

coverage_df = pd.DataFrame(coverage_rows).sort_values("keyword_review_count", ascending=False)
coverage_df

## 6. NLI 모델 로드

제공된 Colab 파일과 같은 모델을 사용합니다. `entailment` label의 softmax probability가 factor relevance입니다.

In [ ]:
MODEL_NAME = "cross-encoder/nli-deberta-v3-large"
NLI_DEVICE = os.environ.get("HB_SCORE_NLI_DEVICE")
if NLI_DEVICE is None:
    NLI_DEVICE = None if is_colab_runtime() else "cpu"
MODEL_BATCH_SIZE = int(os.environ.get("HB_SCORE_MODEL_BATCH_SIZE", "32" if is_colab_runtime() else "4"))
REVIEW_BATCH_SIZE = int(os.environ.get("HB_SCORE_REVIEW_BATCH_SIZE", "32" if is_colab_runtime() else "4"))
MAX_CONTEXT_CHARS = int(os.environ.get("HB_SCORE_MAX_CONTEXT_CHARS", "900"))

print(f"[INFO] Loading {MODEL_NAME} ...")
print(f"[INFO] NLI_DEVICE={NLI_DEVICE or 'auto'} MODEL_BATCH_SIZE={MODEL_BATCH_SIZE} REVIEW_BATCH_SIZE={REVIEW_BATCH_SIZE}")
if NLI_DEVICE:
    nli_model = CrossEncoder(MODEL_NAME, device=NLI_DEVICE)
else:
    nli_model = CrossEncoder(MODEL_NAME)
label_map = {str(label).lower(): int(index) for index, label in nli_model.config.id2label.items()}
ENTAIL_IDX = label_map.get("entailment", 1)
print(f"[INFO] Model loaded. Entailment index: {ENTAIL_IDX}")


## 7. Scoring 대상 선택

`smoke` 모드에서는 빠른 regression test를 위해 일부 리뷰만 계산합니다. `full` 모드에서는 전체 리뷰 98k개를 계산합니다.

In [ ]:
def select_scoring_reviews(df: pd.DataFrame):
    if RUN_MODE == "full":
        return df.copy().reset_index(drop=True)
    selected_parts = []
    for _, place_df in df.sort_values(["place_rank", "review_index"]).groupby("place_id", sort=False):
        if len(selected_parts) >= SMOKE_RESTAURANT_LIMIT:
            break
        selected_parts.append(place_df.head(SMOKE_REVIEWS_PER_RESTAURANT))
    return pd.concat(selected_parts, ignore_index=True)


scoring_reviews_df = select_scoring_reviews(reviews_df)
{
    "run_mode": RUN_MODE,
    "scoring_reviews": int(len(scoring_reviews_df)),
    "scoring_restaurants": int(scoring_reviews_df["place_id"].nunique()),
}

## 8. Review-factor NLI score 계산

각 리뷰와 factor hypothesis를 pair로 만들고 entailment probability를 평균합니다. `factor_distance`는 `1 - factor_relevance`입니다.

In [ ]:
def premise_for_review(row):
    context = normalize_space(row.get("context_text"))
    if not context:
        context = "No review text or structured restaurant experience labels are available."
    context = context[:MAX_CONTEXT_CHARS]
    return f"Focus only on what the reviewer says about the restaurant experience. Review: {context}"


def source_rating_for_factor(row, factor_id):
    label = STRUCTURED_RATING_BY_FACTOR.get(factor_id)
    if label == "음식" and not pd.isna(row.get("food_rating")):
        return float(row.get("food_rating"))
    if label == "서비스" and not pd.isna(row.get("service_rating")):
        return float(row.get("service_rating"))
    if label == "분위기" and not pd.isna(row.get("atmosphere_rating")):
        return float(row.get("atmosphere_rating"))
    rating = row.get("rating")
    if pd.isna(rating):
        return 0.0
    return float(rating)


def score_review_batch(batch_df: pd.DataFrame):
    pairs = []
    order = []
    for row in batch_df.to_dict("records"):
        premise = premise_for_review(row)
        for factor in FACTORS:
            for hypothesis in factor["hypotheses"]:
                pairs.append((premise, hypothesis))
                order.append((row, factor["id"], hypothesis))
    logits = nli_model.predict(pairs, batch_size=MODEL_BATCH_SIZE, show_progress_bar=False)
    probabilities = softmax(logits, axis=-1)
    relevance_values = defaultdict(list)
    hypothesis_values = defaultdict(list)
    for index, (row, factor_id, hypothesis) in enumerate(order):
        key = (row["review_uid"], factor_id)
        entailment_probability = float(probabilities[index][ENTAIL_IDX])
        relevance_values[key].append(entailment_probability)
        hypothesis_values[key].append({"hypothesis": hypothesis, "entailment": entailment_probability})
    rows = []
    row_by_uid = {row["review_uid"]: row for row in batch_df.to_dict("records")}
    for (review_uid, factor_id), values in relevance_values.items():
        row = row_by_uid[review_uid]
        factor_relevance = float(np.mean(values))
        base_score = source_rating_for_factor(row, factor_id)
        rows.append(
            {
                "review_uid": review_uid,
                "source_review_id": row.get("source_review_id"),
                "place_rank": row.get("place_rank"),
                "place_id": row.get("place_id"),
                "place_name": row.get("place_name"),
                "review_index": row.get("review_index"),
                "factor_id": factor_id,
                "factor_label": FACTOR_LABELS[factor_id],
                "source_rating_for_factor": round(base_score, 4),
                "factor_relevance": round(factor_relevance, 6),
                "factor_distance": round(1.0 - factor_relevance, 6),
                "review_hb_score": round(base_score * factor_relevance, 6),
                "hypothesis_entailments": hypothesis_values[(review_uid, factor_id)],
            }
        )
    return pd.DataFrame(rows)


def score_reviews_with_nli(df: pd.DataFrame):
    scored_parts = []
    for start in tqdm(range(0, len(df), REVIEW_BATCH_SIZE), desc="NLI review batches"):
        batch_df = df.iloc[start:start + REVIEW_BATCH_SIZE].copy()
        scored_parts.append(score_review_batch(batch_df))
    return pd.concat(scored_parts, ignore_index=True) if scored_parts else pd.DataFrame()


review_factor_scores_df = score_reviews_with_nli(scoring_reviews_df)
review_factor_scores_df.head()

## 9. 식당별 HB score 집계

Restaurant-level score는 factor별 `review_hb_score` 합을 전체 수집 리뷰 수로 나누고, 리뷰 수가 많은 식당에 capped log bonus를 더합니다.

In [ ]:
COUNT_BONUS_MAX = 0.25


def restaurant_metadata(df: pd.DataFrame):
    columns = [
        "place_rank",
        "place_id",
        "place_name",
        "formatted_address",
        "google_maps_uri",
        "place_rating",
        "user_rating_count",
        "collected_review_count",
        "collection_status",
    ]
    return df.sort_values(["place_rank", "review_index"]).drop_duplicates("place_id")[columns].copy()


def clipped_score(value):
    return round(float(min(5.0, max(0.0, value))), 4)


def build_restaurant_scores(review_scores_df: pd.DataFrame, all_reviews_df: pd.DataFrame):
    metadata_df = restaurant_metadata(all_reviews_df)
    max_popularity_count = float(metadata_df["user_rating_count"].fillna(metadata_df["collected_review_count"]).max())
    grouped = review_scores_df.groupby(["place_id", "factor_id"], dropna=False)
    aggregate_df = grouped.agg(
        review_hb_score_sum=("review_hb_score", "sum"),
        mean_factor_relevance=("factor_relevance", "mean"),
        mean_factor_distance=("factor_distance", "mean"),
        scored_review_count=("review_uid", "nunique"),
    ).reset_index()
    aggregate_by_place_factor = {(row.place_id, row.factor_id): row for row in aggregate_df.itertuples(index=False)}
    restaurants = []
    for row in metadata_df.to_dict("records"):
        popularity_count = row.get("user_rating_count")
        if pd.isna(popularity_count) or not popularity_count:
            popularity_count = row.get("collected_review_count") or 0
        count_bonus = COUNT_BONUS_MAX * math.log1p(float(popularity_count)) / math.log1p(max_popularity_count)
        scores = {}
        for factor in FACTORS:
            aggregate_row = aggregate_by_place_factor.get((row["place_id"], factor["id"]))
            if aggregate_row is None:
                review_hb_score_sum = 0.0
                mean_factor_relevance = 0.0
                mean_factor_distance = 1.0
                scored_review_count = 0
            else:
                review_hb_score_sum = float(aggregate_row.review_hb_score_sum)
                mean_factor_relevance = float(aggregate_row.mean_factor_relevance)
                mean_factor_distance = float(aggregate_row.mean_factor_distance)
                scored_review_count = int(aggregate_row.scored_review_count)
            raw_hb_score = review_hb_score_sum / max(1, int(row["collected_review_count"]))
            scores[factor["id"]] = {
                "factor_label": factor["label"],
                "hb_score": clipped_score(raw_hb_score + count_bonus),
                "raw_hb_score": round(raw_hb_score, 6),
                "count_bonus": round(count_bonus, 6),
                "review_hb_score_sum": round(review_hb_score_sum, 6),
                "mean_factor_relevance": round(mean_factor_relevance, 6),
                "mean_factor_distance": round(mean_factor_distance, 6),
                "scored_review_count": scored_review_count,
                "collected_review_count": int(row["collected_review_count"]),
            }
        restaurants.append(
            {
                "place_rank": int(row["place_rank"]),
                "place_id": row["place_id"],
                "place_name": row["place_name"],
                "formatted_address": row["formatted_address"],
                "google_maps_uri": row["google_maps_uri"],
                "google_place_rating": row["place_rating"],
                "popularity_count": int(popularity_count),
                "collected_review_count": int(row["collected_review_count"]),
                "collection_status": row["collection_status"],
                "scores": scores,
            }
        )
    return restaurants


restaurant_scores = build_restaurant_scores(review_factor_scores_df, reviews_df)
restaurant_scores[:2]

## 10. 산출물 저장

`full` 모드에서는 `datasets/derived/` 아래에 web-app용 JSON과 audit용 CSV.GZ를 저장합니다. `smoke` 모드에서는 같은 파일명을 `/tmp/hidden-bites-hb-score-smoke` 아래에 저장합니다.

In [ ]:
def serializable_factor_schema():
    return [
        {
            "id": factor["id"],
            "label": factor["label"],
            "hypotheses": factor["hypotheses"],
            "structured_rating_label": factor["structured_rating_label"],
        }
        for factor in FACTORS
    ]


ACTIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
restaurant_output = {
    "metadata": {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "run_mode": RUN_MODE,
        "model_name": MODEL_NAME,
        "count_bonus_max": COUNT_BONUS_MAX,
        "source_dataset_dir": str(DATASET_DIR),
        "source_restaurant_count": int(reviews_df["place_id"].nunique()),
        "source_review_count": int(len(reviews_df)),
        "scored_restaurant_count": int(scoring_reviews_df["place_id"].nunique()),
        "scored_review_count": int(scoring_reviews_df["review_uid"].nunique()),
    },
    "factors": serializable_factor_schema(),
    "restaurants": restaurant_scores,
}

restaurant_output_path = ACTIVE_OUTPUT_DIR / "hb-score-restaurants.json"
review_scores_output_path = ACTIVE_OUTPUT_DIR / "hb-score-review-factor-scores.csv.gz"

with restaurant_output_path.open("w", encoding="utf-8") as file:
    json.dump(restaurant_output, file, ensure_ascii=False, indent=2)

review_scores_export_df = review_factor_scores_df.drop(columns=["hypothesis_entailments"])
review_scores_export_df.to_csv(review_scores_output_path, index=False, compression="gzip")

{
    "restaurant_output_path": str(restaurant_output_path),
    "review_scores_output_path": str(review_scores_output_path),
    "restaurant_output_bytes": restaurant_output_path.stat().st_size,
    "review_scores_output_bytes": review_scores_output_path.stat().st_size,
}

## 11. Regression checks

아래 셀은 NLI 모델이 최소한의 기대 동작을 하는지 확인합니다. 맛 리뷰는 `Taste`, 웨이팅 리뷰는 `Wait/Queue` relevance가 높아야 하고, 빈 리뷰는 모든 factor가 낮아야 합니다.

In [ ]:
fixture_reviews_df = pd.DataFrame(
    [
        {
            "review_uid": "fixture-001",
            "source_review_id": "fixture-001",
            "place_rank": 0,
            "place_id": "fixture-place",
            "place_name": "Fixture Restaurant",
            "review_index": 1,
            "rating": 5.0,
            "context_text": "The noodles were flavorful, the broth was deep, and every dish tasted excellent.",
            "food_rating": np.nan,
            "service_rating": np.nan,
            "atmosphere_rating": np.nan,
        },
        {
            "review_uid": "fixture-002",
            "source_review_id": "fixture-002",
            "place_rank": 0,
            "place_id": "fixture-place",
            "place_name": "Fixture Restaurant",
            "review_index": 2,
            "rating": 4.0,
            "context_text": "We waited in line for forty minutes even though we had a reservation, but the queue moved eventually.",
            "food_rating": np.nan,
            "service_rating": np.nan,
            "atmosphere_rating": np.nan,
        },
        {
            "review_uid": "fixture-003",
            "source_review_id": "fixture-003",
            "place_rank": 0,
            "place_id": "fixture-place",
            "place_name": "Fixture Restaurant",
            "review_index": 3,
            "rating": np.nan,
            "context_text": "",
            "food_rating": np.nan,
            "service_rating": np.nan,
            "atmosphere_rating": np.nan,
        },
    ]
)

fixture_scores_df = score_reviews_with_nli(fixture_reviews_df)
fixture_pivot = fixture_scores_df.pivot(index="review_uid", columns="factor_id", values="factor_relevance")

assert fixture_pivot.loc["fixture-001", "taste"] >= 0.45
assert fixture_pivot.loc["fixture-002", "wait_queue"] >= 0.45
assert fixture_scores_df["review_hb_score"].between(0, 5).all()
assert review_factor_scores_df["review_hb_score"].between(0, 5).all()
for restaurant in restaurant_scores:
    for factor_score in restaurant["scores"].values():
        assert 0.0 <= factor_score["hb_score"] <= 5.0

fixture_pivot

## 12. Web-app 사용 메모

`hb-score-restaurants.json`의 `restaurants[].scores`는 factor id를 key로 갖습니다. Web-app에서 dropdown factor id를 선택하면 해당 factor의 `hb_score`를 y-axis 값으로 쓰면 됩니다.

`smoke` 모드 결과는 schema 검증용이므로 실제 ranking으로 해석하면 안 됩니다. 최종 그래프용 데이터는 Colab GPU에서 `full` 모드로 재실행해 생성합니다.